# Prepare Models for Deployment on Streamlit

If using NN or Stack (which uses NN as one of its base models), must import, change device to CPU, then re-export

## Imports

In [34]:
import os

os.environ["PYTORCH_MPS_ENABLE"] = "0"
import torch
import joblib
import sys
import numpy as np
from shutil import rmtree
sys.path.append("../")
from src.config import BASE_PATH
from src.data_utils import get_data, get_models

##Data
OUTCOME_DICT = {
    "surg": get_data("outcome_surg"),
    "bleed": get_data("outcome_bleed"),
    "asp": get_data("outcome_asp"),
    "mort": get_data("outcome_mort"),
}

##Models
model_prefix_list = ["lr"]
model_dir = BASE_PATH / "cal_models"
MODEL_DICT = {}
for outcome in OUTCOME_DICT.keys():
    MODEL_DICT[outcome] = get_models(model_prefix_list, outcome, file_dir=model_dir)

In [36]:
model_output_dir = BASE_PATH / "app" / "models"
if model_output_dir.exists():
    rmtree(model_output_dir)
model_output_dir.mkdir(exist_ok=False, parents=True)
for outcome in MODEL_DICT.keys():
    model = MODEL_DICT[outcome]['lr']
    joblib.dump(model, model_output_dir / f"{outcome}_lr.joblib")

## Change to CPU, ensure consistency, export

In [ ]:
# for outcome, outcome_data in OUTCOME_DICT.items():
#     print(f"Working on {outcome}...")
#     X_val = outcome_data["X_val"]
#     y_val = outcome_data["y_val"]
#     ## Load initial calibrated model + val predictions
#     model = MODEL_DICT[outcome]["stack"]
#     preds_original = model.predict_proba(X_val)[:, 1]

#     # Access the calibrated classifiers (base classifiers)
#     base_estimator = model.calibrated_classifiers_[0].estimator
#     if hasattr(base_estimator, "estimators_"):
#         for j, estimator in enumerate(base_estimator.estimators_):
#             estimator_name = type(estimator).__name__
#             print(f"\t [{j}] {estimator_name}")

#             # Check if it's your TorchNNClassifier
#             if "NN" in estimator_name:
#                 print(f"\t\t → Found PyTorch model with device={estimator.device}")

#                 # Move the internal PyTorch model to CPU
#                 if hasattr(estimator, "model_") and estimator.model_ is not None:
#                     # Save state dict and reload on CPU (clears MPS references)
#                     state_dict = estimator.model_.state_dict()
#                     # Create new model instance on CPU
#                     estimator.model_ = estimator.model_.cpu()
#                     estimator.model_.load_state_dict(
#                         {k: v.cpu() for k, v in state_dict.items()}
#                     )
#                     estimator.model_.eval()
#                     # Clear any lingering MPS tensors
#                     for param in estimator.model_.parameters():
#                         param.data = param.data.cpu()
#                     for buffer in estimator.model_.buffers():
#                         buffer.data = buffer.data.cpu()
#                     print(f"\t\t → Moved model_ to CPU and cleared MPS references")

#                 # Update device attribute
#                 estimator.device = "cpu"
#                 print(f"\t\t → Set device='cpu'")

#     export_path = BASE_PATH / "app" / "models" / f"{outcome}_stack.joblib"
#     if export_path.exists():
#         print(f"\t Over-writing model at path {export_path}")
#         export_path.unlink()
#     export_path.parent.mkdir(exist_ok=True, parents=True)
#     joblib.dump(model, export_path)
#     print(f"\t\t → Exported model!")
#     print(f"\t Checking that MPS and CPU models behave ~same...")
#     model = joblib.load(export_path)
#     preds_cpu = model.predict_proba(X_val)[:, 1]

#     # Compare
#     diff = np.abs(preds_original - preds_cpu)
#     print(f"\t Max difference: {diff.max()}")  # Should be ~1e-7 (floating point noise)
#     print(f"\t Mean difference: {diff.mean()}")  # Should be ~0

#     assert np.allclose(preds_original, preds_cpu, rtol=1e-5)
#     print("\t ✓ Predictions are identical!")
#     ## Ensure CPU
#     for name, param in estimator.model_.named_parameters():
#         print(f"{name}: {param.device}")